# MAEPiMS 2026 — Final Short Colab Solution

**Final framework: SAIP (Seasonal Analog + Intensity + Propagation)**

The final forecast is built from the supplied three-season synthetic data. State-week forecasts use 45% of the latest observed season and 55% of the preceding season. Historical pseudo-seasonal holdouts are used for validation and uncertainty estimation.

In [ ]:
# 1. Libraries
!pip -q install xgboost openpyxl reportlab

import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

POP = 229501935

In [ ]:
# 2. Upload ZIP
from google.colab import files

uploaded = files.upload()
zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file) as z:
    z.extractall("/content/data")

In [ ]:
# 3. Load data
from pathlib import Path

folder = Path("/content/data")

national = pd.read_csv(list(folder.rglob("nigeria_flu_weekly_national.csv"))[0])
state = pd.read_csv(list(folder.rglob("nigeria_flu_weekly_by_state.csv"))[0])
metadata = pd.read_csv(list(folder.rglob("nigeria_flu_state_metadata.csv"))[0])
season_summary = pd.read_csv(list(folder.rglob("nigeria_flu_season_summary.csv"))[0])

national["week_start"] = pd.to_datetime(national["week_start"])
state["week_start"] = pd.to_datetime(state["week_start"])

national = national.rename(columns={
    "hospitalizations_per_100k":"hosp_per_100k"
})

print(national.shape)
print(state.shape)
print(metadata.shape)

In [ ]:
# 4. Check data
print("States:", state.state.nunique())
print("Seasons:", state.season.unique())
print("Missing values:", state.isna().sum().sum())
print("Duplicates:", state.duplicated(
    ["season","state","epi_week_of_season"]
).sum())

## 1. Prepare the model

In [ ]:
# 5. Add metadata and regions
state = state.merge(
    metadata[["state","hc_access"]],
    on="state", how="left"
)

state["region"] = np.where(
    state.zone.isin(["NW","NE","NC"]),
    "North", "South"
)

In [ ]:
# 6. Sort data
state = state.sort_values(
    ["state","season","epi_week_of_season"]
).reset_index(drop=True)

## 2. Final forecasting rule

In [ ]:
# 7. Seasonal forecast function

def forecast_season(history, target, test_season):
    years = sorted(history.season.unique())
    year = int(test_season[:4])
    prev = [s for s in years if int(s[:4]) < year]

    recent = prev[-1]
    a = history[history.season == recent][
        ["state","epi_week_of_season",target]
    ].rename(columns={target:"recent"})

    if len(prev) == 1:
        a["pred"] = a["recent"]
        return a[["state","epi_week_of_season","pred"]]

    older = prev[-2]
    b = history[history.season == older][
        ["state","epi_week_of_season",target]
    ].rename(columns={target:"older"})

    out = a.merge(
        b, on=["state","epi_week_of_season"], how="left"
    )
    out["pred"] = (
        0.45*out.recent +
        0.55*out.older.fillna(out.recent)
    )

    return out[["state","epi_week_of_season","pred"]]

## 3. Historical validation

In [ ]:
# 8. Backtest two seasons
all_val = []

for season in ["2024/2025","2025/2026"]:
    train = state[state.season.str[:4].astype(int) < int(season[:4])]
    test = state[state.season == season]

    for target in [
        "cases_per_100k",
        "hosp_per_100k",
        "deaths_per_100k"
    ]:
        p = forecast_season(train, target, season)

        x = test[
            ["state","epi_week_of_season",target,"population","region"]
        ].merge(
            p,
            on=["state","epi_week_of_season"],
            how="inner"
        )

        x["target"] = target
        x["season"] = season
        all_val.append(x)

validation_data = pd.concat(all_val, ignore_index=True)
print(validation_data.shape)

In [ ]:
# 9. Validation metrics
rows = []

for (season,target), g in validation_data.groupby(["season","target"]):
    rows.append({
        "season": season,
        "target": target,
        "MAE": mean_absolute_error(g[target], g.pred),
        "RMSE": mean_squared_error(g[target], g.pred)**0.5
    })

validation = pd.DataFrame(rows)
display(validation.round(4))

In [ ]:
# 10. National validation
national_val = []

for (season,target), g in validation_data.groupby(["season","target"]):
    w = g.groupby("epi_week_of_season").apply(
        lambda x: pd.Series({
            "actual": np.average(x[target], weights=x.population),
            "pred": np.average(x.pred, weights=x.population)
        }), include_groups=False
    ).reset_index()

    rows = {
        "season": season,
        "target": target,
        "MAE": mean_absolute_error(w.actual, w.pred),
        "RMSE": mean_squared_error(w.actual, w.pred)**0.5,
        "actual_peak_week": int(w.loc[w.actual.idxmax(),"epi_week_of_season"]),
        "pred_peak_week": int(w.loc[w.pred.idxmax(),"epi_week_of_season"]),
        "peak_week_error": abs(
            int(w.loc[w.actual.idxmax(),"epi_week_of_season"])
            - int(w.loc[w.pred.idxmax(),"epi_week_of_season"])
        ),
        "cumulative_error_pct": abs(w.pred.sum()-w.actual.sum())/w.actual.sum()*100
    }
    rows["peak_incidence_error_pct"] = abs(
        w.pred.max()-w.actual.max()
    )/w.actual.max()*100
    national_val.append(rows)

national_validation = pd.DataFrame(national_val)
display(national_validation.round(3))

## 4. Probabilistic evaluation

In [ ]:
# 11. Metric functions

def pinball(y, p, q):
    e = np.asarray(y) - np.asarray(p)
    return np.mean(np.maximum(q*e, (q-1)*e))


def wis(y, med, lo50, hi50, lo90, hi90):
    y = np.asarray(y)

    s50 = (hi50-lo50) + 4*np.maximum(lo50-y,0) + 4*np.maximum(y-hi50,0)
    s90 = (hi90-lo90) + 20*np.maximum(lo90-y,0) + 20*np.maximum(y-hi90,0)

    return np.mean(
        0.5*np.abs(y-med) + 0.25*s50 + 0.25*s90
    )


def crps(y, q, pred):
    loss = []
    for i in range(len(q)):
        e = y-pred[:,i]
        loss.append(np.maximum(q[i]*e,(q[i]-1)*e))
    loss = np.array(loss).T
    return 2*np.trapezoid(loss,q,axis=1).mean()

In [ ]:
# 12. Historical residuals
residuals = {}

for target in ["cases_per_100k","hosp_per_100k","deaths_per_100k"]:
    residuals[target] = (
        validation_data.loc[
            validation_data.target == target, target
        ]
        - validation_data.loc[
            validation_data.target == target, "pred"
        ]
    ).values

In [ ]:
# 13. Coverage and scoring
prob_rows = []

for target in residuals:
    r = residuals[target]
    q05,q25,q75,q95 = np.quantile(r,[.05,.25,.75,.95])

    d = validation_data[validation_data.target == target].copy()
    d["q50"] = d.pred
    d["q05"] = np.maximum(d.pred+q05,0)
    d["q25"] = np.maximum(d.pred+q25,0)
    d["q75"] = d.pred+q75
    d["q95"] = d.pred+q95

    y = d[target].values
    P = d[["q05","q25","q50","q75","q95"]].values

    prob_rows.append({
        "target": target,
        "WIS": wis(y,d.q50,d.q25,d.q75,d.q05,d.q95),
        "CRPS": crps(y,np.array([.05,.25,.5,.75,.95]),P),
        "QL05": pinball(y,d.q05,.05),
        "QL25": pinball(y,d.q25,.25),
        "QL50": pinball(y,d.q50,.50),
        "QL75": pinball(y,d.q75,.75),
        "QL95": pinball(y,d.q95,.95),
        "coverage50": ((y>=d.q25)&(y<=d.q75)).mean(),
        "coverage90": ((y>=d.q05)&(y<=d.q95)).mean()
    })

prob_results = pd.DataFrame(prob_rows)
display(prob_results.round(4))

## 5. Final 2026/2027 forecast

In [ ]:
# 14. Forecast all targets
fc = state[[
    "state","zone","region","population","epi_week_of_season"
]].drop_duplicates().copy()

for target,name in [
    ("cases_per_100k","cases"),
    ("hosp_per_100k","hospitalizations"),
    ("deaths_per_100k","deaths")
]:
    p = forecast_season(state,target,"2026/2027")
    p = p.rename(columns={"pred":name+"_per_100k"})
    fc = fc.merge(
        p,
        on=["state","epi_week_of_season"],
        how="left"
    )

for name in ["cases","hospitalizations","deaths"]:
    fc[name] = fc[name+"_per_100k"]*fc.population/100000

fc["season"] = "2026/2027"

display(fc.head())

In [ ]:
# 15. National forecast
nat = fc.groupby(
    ["season","epi_week_of_season"], as_index=False
).agg(
    predicted_cases=("cases","sum"),
    predicted_hospitalizations=("hospitalizations","sum"),
    predicted_deaths=("deaths","sum")
)

nat["cases_per_100k"] = nat.predicted_cases/POP*100000
nat["hosp_per_100k"] = nat.predicted_hospitalizations/POP*100000
nat["deaths_per_100k"] = nat.predicted_deaths/POP*100000

nat["week_start"] = pd.date_range(
    pd.to_datetime(state[state.season=="2025/2026"].week_start.min())
    + pd.Timedelta(days=364),
    periods=52, freq="7D"
)

display(nat.head())

In [ ]:
# 16. Add national quantiles
for name,rate in [
    ("cases","cases_per_100k"),
    ("hospitalizations","hosp_per_100k"),
    ("deaths","deaths_per_100k")
]:
    q05,q25,q75,q95 = np.quantile(
        residuals[rate if rate in residuals else {
            "cases":"cases_per_100k",
            "hospitalizations":"hosp_per_100k",
            "deaths":"deaths_per_100k"
        }[name]],
        [.05,.25,.75,.95]
    )

    nat[f"{name}_q50"] = nat[rate]
    nat[f"{name}_q05"] = np.maximum(nat[rate]+q05,0)
    nat[f"{name}_q25"] = np.maximum(nat[rate]+q25,0)
    nat[f"{name}_q75"] = nat[rate]+q75
    nat[f"{name}_q95"] = nat[rate]+q95

    for q in ["q05","q25","q50","q75","q95"]:
        nat[f"{name}_{q}_count"] = nat[f"{name}_{q}"]*POP/100000

print("Quantiles added.")

## 6. Seasonal targets

In [ ]:
# 17. Seasonal targets
rows=[]

for name,col,rate in [
    ("cases","predicted_cases","cases_per_100k"),
    ("hospitalizations","predicted_hospitalizations","hosp_per_100k"),
    ("deaths","predicted_deaths","deaths_per_100k")
]:
    i = nat[rate].idxmax()
    rows.append({
        "target": name,
        "peak_week": int(nat.loc[i,"epi_week_of_season"]),
        "peak_incidence": nat.loc[i,rate],
        "peak_count": nat.loc[i,col],
        "cumulative": nat[col].sum()
    })

targets = pd.DataFrame(rows)
targets["attack_rate_pct"] = np.nan
targets.loc[targets.target=="cases","attack_rate_pct"] = (
    targets.loc[targets.target=="cases","cumulative"].iloc[0]/POP*100
)

display(targets.round(2))

## 7. Regional forecasts

In [ ]:
# 18. North and South
regional = fc.groupby(
    ["season","region","epi_week_of_season"], as_index=False
).agg(
    cases=("cases","sum"),
    hospitalizations=("hospitalizations","sum"),
    deaths=("deaths","sum"),
    population=("population","sum")
)

regional["cases_per_100k"] = regional.cases/regional.population*100000
regional["hosp_per_100k"] = regional.hospitalizations/regional.population*100000
regional["deaths_per_100k"] = regional.deaths/regional.population*100000

display(regional.groupby("region").agg(
    peak_week=("cases_per_100k",lambda x: regional.loc[x.idxmax(),"epi_week_of_season"]),
    cumulative_incidence=("cases_per_100k","sum")
).reset_index())

## 8. State-level results

In [ ]:
# 19. State summaries
state_summary = fc.groupby("state",as_index=False).agg(
    cumulative_cases=("cases","sum"),
    cumulative_hospitalizations=("hospitalizations","sum"),
    cumulative_deaths=("deaths","sum"),
    population=("population","first"),
    region=("region","first"),
    zone=("zone","first")
)

state_summary["attack_rate_pct"] = (
    state_summary.cumulative_cases/
    state_summary.population*100
)

state_summary["death_rate_per_100k"] = (
    state_summary.cumulative_deaths/
    state_summary.population*100000
)

display(state_summary.sort_values(
    "cumulative_cases",ascending=False
).head(10))

## 9a. State-level case intervals
The same historical case residual distribution is used for transparent state-level uncertainty bands.


In [ ]:
# 19b. State case intervals
r = residuals["cases_per_100k"]
q05, q25, q75, q95 = np.quantile(r, [.05,.25,.75,.95])

fc["cases_q50"] = fc.cases_per_100k
fc["cases_q05"] = np.maximum(fc.cases_q50 + q05, 0)
fc["cases_q25"] = np.maximum(fc.cases_q50 + q25, 0)
fc["cases_q75"] = fc.cases_q50 + q75
fc["cases_q95"] = fc.cases_q50 + q95

for q in ["q05","q25","q50","q75","q95"]:
    fc[f"cases_{q}_count"] = fc[f"cases_{q}"]*fc.population/100000

print("State case intervals added.")

## 9b. Optional peak-window probability


In [ ]:
# 19c. Optional probability that peak is in a week window
rng = np.random.default_rng(42)
R = residuals["cases_per_100k"]
base = nat.cases_per_100k.to_numpy()

sim = base[None,:] + rng.choice(R, size=(10000,52), replace=True)
peaks = sim.argmax(axis=1) + 1

for lo, hi in [(14,16),(15,17),(16,18)]:
    p = np.mean((peaks >= lo) & (peaks <= hi))
    print(f"P(peak weeks {lo}-{hi}): {p:.3f}")

## 9. Optional transmission probabilities

In [ ]:
# 20. Simple empirical transmission probabilities
nat["increase"] = nat.cases_per_100k.diff() > 0
nat["decrease"] = nat.cases_per_100k.diff() < 0

print("Probability increasing:", round(nat.increase.mean(),3))
print("Probability decreasing:", round(nat.decrease.mean(),3))

## 10. Figures

In [ ]:
# 21. National forecast
plt.figure(figsize=(11,5))
plt.plot(nat.epi_week_of_season,nat.cases_q50_count,label="Median")
plt.fill_between(
    nat.epi_week_of_season,
    nat.cases_q05_count,
    nat.cases_q95_count,
    alpha=.2,label="90% interval"
)
plt.xlabel("Epidemiological week")
plt.ylabel("Weekly cases")
plt.title("2026/2027 National Influenza Forecast")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig("MAEPiMS_national_forecast.png",dpi=300)
plt.show()

In [ ]:
# 22. North vs South
plt.figure(figsize=(11,5))
for region,g in regional.groupby("region"):
    plt.plot(g.epi_week_of_season,g.cases_per_100k,label=region)
plt.xlabel("Week")
plt.ylabel("Cases per 100,000")
plt.title("North vs South Forecast")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig("MAEPiMS_north_south.png",dpi=300)
plt.show()

In [ ]:
# 23. State heatmap
heat = fc.pivot(
    index="state",
    columns="epi_week_of_season",
    values="cases_per_100k"
)

plt.figure(figsize=(14,10))
plt.imshow(heat,aspect="auto")
plt.colorbar(label="Cases per 100,000")
plt.xlabel("Week")
plt.ylabel("State")
plt.title("State-level Weekly Forecast")
plt.tight_layout()
plt.savefig("MAEPiMS_state_heatmap.png",dpi=300)
plt.show()

## 11. Final files

In [ ]:
# 24. Save tables
national_out = nat.copy()
state_out = fc.copy()
regional_out = regional.copy()

national_out.to_csv("MAEPiMS_national_final.csv",index=False)
state_out.to_csv("MAEPiMS_state_final.csv",index=False)
regional_out.to_csv("MAEPiMS_regional_final.csv",index=False)
targets.to_csv("MAEPiMS_seasonal_targets.csv",index=False)
validation.to_csv("MAEPiMS_validation.csv",index=False)
national_validation.to_csv("MAEPiMS_national_validation.csv",index=False)
prob_results.to_csv("MAEPiMS_probabilistic_validation.csv",index=False)
state_summary.to_csv("MAEPiMS_state_summary.csv",index=False)

with pd.ExcelWriter("MAEPiMS_all_results.xlsx") as writer:
    targets.to_excel(writer,sheet_name="Seasonal",index=False)
    nat.to_excel(writer,sheet_name="National",index=False)
    regional.to_excel(writer,sheet_name="Regional",index=False)
    state_summary.to_excel(writer,sheet_name="State",index=False)
    validation.to_excel(writer,sheet_name="Validation",index=False)
    national_validation.to_excel(writer,sheet_name="National_Validation",index=False)
    prob_results.to_excel(writer,sheet_name="Probabilistic",index=False)

print("All results saved.")

In [ ]:
# 24b. Final QA
print("National weeks:", len(nat))
print("States:", fc.state.nunique())
print("State-week rows:", len(fc))
print("National cumulative cases:", round(nat.predicted_cases.sum()))
print("National cumulative hospitalisations:", round(nat.predicted_hospitalizations.sum()))
print("National cumulative deaths:", round(nat.predicted_deaths.sum()))
print("Attack rate (%):", round(nat.predicted_cases.sum()/POP*100,2))

In [ ]:
# 25. Final ZIP and download
final_files = [
    "MAEPiMS_national_final.csv",
    "MAEPiMS_state_final.csv",
    "MAEPiMS_regional_final.csv",
    "MAEPiMS_seasonal_targets.csv",
    "MAEPiMS_validation.csv",
    "MAEPiMS_national_validation.csv",
    "MAEPiMS_probabilistic_validation.csv",
    "MAEPiMS_state_summary.csv",
    "MAEPiMS_all_results.xlsx",
    "MAEPiMS_national_forecast.png",
    "MAEPiMS_north_south.png",
    "MAEPiMS_state_heatmap.png"
]

with zipfile.ZipFile(
    "MAEPiMS_FINAL_RESULTS.zip","w",zipfile.ZIP_DEFLATED
) as z:
    for f in final_files:
        if os.path.exists(f):
            z.write(f)

from google.colab import files
files.download("MAEPiMS_FINAL_RESULTS.zip")